# Level 1 TCN — Masked Multi-Scale Temporal Convolutional Network

This notebook implements and tests the **Level 1 Masked Multi-Scale TCN** architecture designed to overcome the padding, scaling, and temporal collapse issues that caused earlier deep learning models to underperform the Random Forest baseline.

**Key Features of this Architecture:**
- **Masked Temporal Convolutions:** Ignores padded timesteps properly.
- **Per-channel Normalization:** Handles the vastly different scales of IMU, ToF, and Thermopile sensors.
- **Residual Dilated Blocks:** Captures multi-scale temporal dynamics without losing gradient flow.
- **Masked Mean/Max Pooling:** Aggregates sequence features without diluting short sequences with padding zeros.

Local quick runs use `data/sample.csv`. Set `use_sample_data = False` for full `train.csv`.
Switch `search_mode` between `'grid'` and `'bayesian'` in the config cell.

In [1]:
import os
import sys
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

# Suppress TF and general warnings for cleaner output
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
warnings.filterwarnings('ignore')

# Local path routing (works from notebooks/ or project root)
current_dir = os.getcwd()
workspace_root = current_dir
if os.path.basename(current_dir) == 'notebooks':
    workspace_root = os.path.dirname(current_dir)

src_path = os.path.join(workspace_root, 'src')
sys.path.insert(0, workspace_root)
sys.path.insert(0, src_path)

# Kaggle path routing
try:
    dataset_name = os.listdir('/kaggle/input/datasets/keithmarange')[0]
    sys.path.append(f'/kaggle/input/datasets/keithmarange/{dataset_name}/')
    sys.path.append(f'/kaggle/input/datasets/keithmarange/{dataset_name}/src')
except Exception:
    pass

print('Paths configured. Workspace root:', workspace_root)

Paths configured. Workspace root: c:\Users\maran\OneDrive\Documents\Git Profile\cmi_dexter


In [2]:
import tensorflow as tf
from sklearn.model_selection import GridSearchCV, GroupKFold, GroupShuffleSplit
from sklearn.metrics import f1_score, make_scorer

try:
    from skopt import BayesSearchCV
    from skopt.space import Categorical, Integer, Real
    SKOPT_AVAILABLE = True
except ImportError:
    BayesSearchCV = None
    Categorical = Integer = Real = None
    SKOPT_AVAILABLE = False

try:
    from src import data_utils
    from src.base_utils_qwen import (
        SequenceExtractor,
        competition_scorer,
        evaluate_holdout,
        make_competition_scorer,
        prepare_bayesian_space,
    )
    from src.tcn_level_one import MaskedMultiScaleTCNClassifier
    print('Imports loaded from src/')
except ImportError:
    import data_utils
    from base_utils_qwen import (
        SequenceExtractor,
        competition_scorer,
        evaluate_holdout,
        make_competition_scorer,
        prepare_bayesian_space,
    )
    from tcn_level_one import MaskedMultiScaleTCNClassifier
    print('Imports loaded from flat src path')

# Install Bayesian search dependency if missing (safe to re-run)
if not SKOPT_AVAILABLE:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'scikit-optimize'])
    from skopt import BayesSearchCV
    from skopt.space import Categorical, Integer, Real
    SKOPT_AVAILABLE = True
    print('Installed scikit-optimize')
else:
    import skopt
    print(f'scikit-optimize {skopt.__version__} ready')

Custom modules loaded successfully.
scikit-optimize ready for Bayesian search.


## 1. Configuration

Set your target, data source, and search parameters here.

In [ ]:
TARGET_COL = 'bfrb'

# Use data/sample.csv for fast local runs; set False for full train.csv
use_sample_data = False
sample_file = 'sample.csv'

search_mode = 'bayesian'  # 'grid' or 'bayesian'
random_state = 42
n_splits = 1          # 1 -> GroupShuffleSplit; >=2 -> GroupKFold
cv_test_size = 0.3
train_size = 0.7
n_iter = 10         # Bayesian iterations (raise for full runs)
verbose = 4

results_dir = Path('results_tcn_level1')
results_dir.mkdir(exist_ok=True)
timestamp = datetime.now().strftime('%Y%m%d_%H%M')

if TARGET_COL == 'bfrb':
    scorer = competition_scorer
else:
    scorer = make_scorer(f1_score, average='macro', zero_division=0)

search_mode = str(search_mode).lower()
if search_mode in ('bayes', 'bayesian'):
    search_mode = 'bayesian'
elif search_mode != 'grid':
    raise ValueError("search_mode must be 'grid' or 'bayesian'")

if n_splits <= 1:
    cv_object = GroupShuffleSplit(
        n_splits=1,
        test_size=cv_test_size,
        random_state=random_state,
    )
else:
    cv_object = GroupKFold(n_splits=n_splits)

print(f'Configuration set. Target: {TARGET_COL}, Search Mode: {search_mode}')

Configuration set. Target: bfrb, Search Mode: grid


## 2. Data Loading & Preprocessing

Load the raw row-level sensor data and engineer the target variables.

In [4]:
data_root = data_utils.find_data_root()
sample_path = data_root / sample_file

if use_sample_data and sample_path.exists():
    raw_train_df = pd.read_csv(sample_path)
    print(f'Using {sample_file}: {raw_train_df["sequence_id"].nunique()} sequences')
else:
    raw_train_df = pd.read_csv(data_root / 'train.csv')
    print(f'Using train.csv: {raw_train_df["sequence_id"].nunique()} sequences')

# Handle demographics if available
demo_path = data_root / 'train_demographics.csv'
if demo_path.exists():
    train_demo_df = pd.read_csv(demo_path)

train_df = raw_train_df.set_index('row_id').copy(deep=True)

# Feature engineering for targets
train_df['gesture'] = train_df['gesture'].fillna('non_bfrb').astype(str)
train_df['orientation'] = train_df['orientation'].fillna('Unknown').astype(str)
train_df['is_target'] = train_df['sequence_type'].eq('Target').astype(int)
train_df['bfrb'] = train_df['gesture'].where(train_df['is_target'].astype(bool), 'non_bfrb')

print(f'Data loaded. Shape: {train_df.shape}')
print(f'Target distribution:\n{train_df[TARGET_COL].value_counts()}')

✅ Found data folder: C:\Users\maran\OneDrive\Documents\Git Profile\cmi_dexter\data
Using sample.csv: 37 sequences
Data loaded. Shape: (2679, 342)
Target distribution:
bfrb
non_bfrb                    958
Neck - pinch skin           365
Forehead - pull hairline    334
Above ear - pull hair       251
Neck - scratch              217
Eyelash - pull hair         191
Forehead - scratch          154
Cheek - pinch skin          110
Eyebrow - pull hair          99
Name: count, dtype: int64


## 3. Train / Holdout Split

We split by `sequence_id` to prevent temporal/sequence leakage.

In [5]:
try:
    train_sample_df, hold_out_df = data_utils.sample_balanced_split(
        train_df,
        train_pct=train_size,
        test_pct=min(0.2, 1 - train_size),
        random_state=random_state,
    )
except Exception as e:
    print(f'Balanced split failed ({e}), falling back to GroupShuffleSplit.')
    seq_df = train_df[['sequence_id', 'is_target', TARGET_COL]].drop_duplicates('sequence_id').sort_values('sequence_id')
    gss = GroupShuffleSplit(n_splits=1, train_size=train_size, random_state=random_state)
    train_idx, test_idx = list(gss.split(seq_df, groups=seq_df['sequence_id']))[0]
    
    train_seqs = seq_df.iloc[train_idx]['sequence_id']
    test_seqs = seq_df.iloc[test_idx]['sequence_id']
    
    train_sample_df = train_df[train_df['sequence_id'].isin(train_seqs)]
    hold_out_df = train_df[train_df['sequence_id'].isin(test_seqs)]

# For TCN, X is the raw row-level DataFrame, y contains the sequence-level metadata
X_train = train_sample_df.copy()
X_test = hold_out_df.copy()

y_train = X_train[['sequence_id', 'is_target', TARGET_COL]].copy()
y_test = X_test[['sequence_id', 'is_target', TARGET_COL]].copy()

groups = X_train['sequence_id'].astype(str)

print('Train sequences:', X_train['sequence_id'].nunique())
print('Test sequences:', X_test['sequence_id'].nunique())

Balanced split failed (Percentage too small. Min viable pct for this data: 1340.5%), falling back to GroupShuffleSplit.
Train sequences: 25
Test sequences: 12


## 4. Model Definition

Initialize the Level 1 Masked Multi-Scale TCN. 

*Note: For smoke testing with `sample.csv`, we keep `epochs` low. Increase `epochs` to 50-100 for full training.*

In [6]:
extractor = SequenceExtractor(
    acc_modes='raw|velocity|jerk',
    rotation_modes='quaternion|angular_velocity',
    tof_modes='pooled_stats|sensor_stats',
    thm_modes='centered_diff',
    motion_filter_mode=None,
    use_dead_reckoning=False,
    compute_dt=True,
    interp_mode='linear',
    padding_value=0.0,
    chunk_window_size=None,
    chunk_stride=None,
    output_format='chunks',
    add_global_context=False,
    resample_modalities=False,
    maxlen=200,
)

model = MaskedMultiScaleTCNClassifier(
    primary_target=TARGET_COL,
    sequence_col='sequence_id',
    extractor=extractor,
    filters=64,
    num_blocks=3,
    kernel_size=3,
    dilations=(1, 2, 4),
    dropout=0.2,
    learning_rate=1e-3,
    batch_size=32,
    epochs=5,                    # Smoke test epochs. Change to 50+ for full runs.
    patience=3,
    validation_split=0.15,
    random_state=random_state,
    verbose=verbose,
    class_weight='balanced',
)

print('Level 1 TCN Model initialized.')

Level 1 TCN Model initialized.


## 5. Parameter Spaces

Define the search spaces for Grid and Bayesian optimization.

In [ ]:
# ============================================================
# PARAMETER SPACE — GRID vs BAYESIAN
# ============================================================

_SKOPT_AVAILABLE = globals().get('SKOPT_AVAILABLE', False)
_ACTIVE_SEARCH_MODE = str(
    globals().get('SEARCH_MODE', globals().get('search_mode', 'grid'))
).lower()
if _ACTIVE_SEARCH_MODE in ('bayes', 'bayesian'):
    _ACTIVE_SEARCH_MODE = 'bayesian'

# Fast grid for sample.csv smoke tests
GRID_PARAM_SPACE_QUICK = {
    'extractor__acc_modes': ['raw|velocity|jerk'],
    'extractor__rotation_modes': ['quaternion|angular_velocity'],
    'extractor__tof_modes': ['pooled_stats|sensor_stats'],
    'extractor__thm_modes': ['centered_diff'],
    'extractor__motion_filter_mode': [None],
    'extractor__use_dead_reckoning': [False],
    'extractor__dead_reckoning_detrend': [False],
    'extractor__kalman_process_noise': [1e-3],
    'extractor__kalman_measurement_noise': [1e-1],
    'extractor__window_size': [7],
    'extractor__smooth_alpha': [None],
    'extractor__clip_value': [None],
    'extractor__interp_mode': ['linear'],
    'extractor__output_format': ['chunks'],
    'extractor__padding_value': [0.0],
    'extractor__maxlen': [160, 200],
    'extractor__chunk_window_size': [None],
    'extractor__chunk_stride': [None],
    'extractor__add_global_context': [False],
    'extractor__resample_modalities': [False],
    'extractor__compute_dt': [True],
    'extractor__imu_native_sampling_rate': [20],
    'extractor__rot_native_sampling_rate': [20],
    'extractor__tof_native_sampling_rate': [5],
    'extractor__thm_native_sampling_rate': [5],
    'filters': [32, 64],
    'num_blocks': [2, 3],
    'kernel_size': [3],
    'dilations': [(1, 2, 4), (1, 2, 4, 8)],
    'dropout': [0.2, 0.3],
    'learning_rate': [1e-3, 1e-4],
    'batch_size': [32],
}

# Practical compact grid for full train.csv
GRID_PARAM_SPACE = {
    'extractor__acc_modes': [
        'raw|velocity|jerk',
        'smoothed|velocity|displacement|jerk',
    ],
    'extractor__rotation_modes': [
        'quaternion|angular_velocity',
        'quaternion|euler|angular_velocity',
    ],
    'extractor__tof_modes': [
        'pooled_stats|sensor_stats',
        'pooled_stats',
    ],
    'extractor__thm_modes': [
        'centered_diff',
        'centered',
    ],
    'extractor__motion_filter_mode': [None],
    'extractor__use_dead_reckoning': [False],
    'extractor__dead_reckoning_detrend': [False],
    'extractor__kalman_process_noise': [1e-3],
    'extractor__kalman_measurement_noise': [1e-1],
    'extractor__window_size': [7],
    'extractor__smooth_alpha': [None],
    'extractor__clip_value': [None],
    'extractor__interp_mode': ['linear'],
    'extractor__output_format': ['chunks'],
    'extractor__padding_value': [0.0],
    'extractor__maxlen': [160, 200],
    'extractor__chunk_window_size': [None, 80],
    'extractor__chunk_stride': [None, 20],
    'extractor__add_global_context': [False],
    'extractor__resample_modalities': [False],
    'extractor__compute_dt': [True],
    'extractor__imu_native_sampling_rate': [20],
    'extractor__rot_native_sampling_rate': [20],
    'extractor__tof_native_sampling_rate': [5],
    'extractor__thm_native_sampling_rate': [5],
    'filters': [64, 128],
    'num_blocks': [3, 4],
    'kernel_size': [3, 5],
    'dilations': [(1, 2, 4), (1, 2, 4, 8), (1, 4, 16)],
    'dropout': [0.1, 0.2, 0.3],
    'learning_rate': [1e-3, 5e-4, 1e-4],
    'batch_size': [32, 64],
}

if _SKOPT_AVAILABLE:
    try:
        BAYESIAN_PARAM_SPACE = {
            'extractor__acc_modes': Categorical([
                'raw',
                'raw|velocity',
                'raw|velocity|jerk',
                'raw|velocity|displacement',
                'raw|velocity|displacement|jerk',
                'smoothed|velocity|jerk',
                'smoothed|velocity|displacement|jerk',
            ]),
            'extractor__rotation_modes': Categorical([
                'quaternion',
                'quaternion|euler',
                'quaternion|angular_velocity',
                'quaternion|euler|angular_velocity',
                'quaternion|delta_euler|angular_velocity',
                'quaternion|angular_velocity|delta_euler',
                'quaternion|angular_velocity|delta_euler|rot6d',
            ]),
            'extractor__tof_modes': Categorical([
                'sensor_stats',
                'pooled_stats',
                'pooled_stats|sensor_stats',
            ]),
            'extractor__thm_modes': Categorical([
                'centered',
                'diff',
                'centered_diff',
            ]),
            'extractor__motion_filter_mode': Categorical([
                None,
                'kalman',
                'extended_kalman',
            ]),
            'extractor__use_dead_reckoning': Categorical([False, True]),
            'extractor__dead_reckoning_detrend': Categorical([False, True]),
            'extractor__kalman_process_noise': Real(1e-5, 1e-1, prior='log-uniform'),
            'extractor__kalman_measurement_noise': Real(1e-3, 1e1, prior='log-uniform'),
            'extractor__window_size': Integer(3, 50),
            'extractor__smooth_alpha': Categorical([
                None, 0.05, 0.10, 0.20, 0.30, 0.50, 0.70, 0.90,
            ]),
            'extractor__clip_value': Categorical([None, 100.0, 150.0]),
            'extractor__interp_mode': Categorical(['linear', 'ffill']),
            'extractor__output_format': Categorical(['chunks']),
            'extractor__padding_value': Categorical([0.0]),
            'extractor__maxlen': Categorical([160, 30, 60, 100, 200]),
            'extractor__chunk_window_size': Categorical([None, 30, 50, 80, 120]),
            'extractor__chunk_stride': Categorical([None, 10, 15, 20, 30, 60]),
            'extractor__add_global_context': Categorical([False, True]),
            'extractor__compute_dt': Categorical([False, True]),
            'extractor__imu_native_sampling_rate': Categorical([20, 100]),
            'extractor__rot_native_sampling_rate': Categorical([20, 100]),
            'extractor__tof_native_sampling_rate': Categorical([5]),
            'extractor__thm_native_sampling_rate': Categorical([5]),
            'extractor__imu_target_sampling_rate': Categorical([20, 100]),
            'extractor__rot_target_sampling_rate': Categorical([20, 100]),
            'extractor__tof_target_sampling_rate': Categorical([5, 10, 20]),
            'extractor__thm_target_sampling_rate': Categorical([5, 10, 20]),
            'extractor__resample_modalities': Categorical([True]),
            'filters': Integer(16, 256),
            'num_blocks': Integer(2, 8),
            'kernel_size': Categorical([3, 5, 7]),
            'dilations': Categorical([(1, 2, 4), (1, 2, 4, 8), (1, 4, 16)]),
            'dropout': Real(0.1, 0.6),
            'learning_rate': Real(1e-3, 1e-2, prior='log-uniform'),
            'batch_size': Categorical([16, 32, 64]),
        }
    except Exception:
        BAYESIAN_PARAM_SPACE = GRID_PARAM_SPACE
else:
    BAYESIAN_PARAM_SPACE = GRID_PARAM_SPACE

if _ACTIVE_SEARCH_MODE == 'bayesian' and _SKOPT_AVAILABLE:
    param_space = BAYESIAN_PARAM_SPACE
    try:
        param_space = prepare_bayesian_space(param_space)
    except Exception:
        pass
elif use_sample_data:
    param_space = GRID_PARAM_SPACE_QUICK
else:
    param_space = GRID_PARAM_SPACE

if _ACTIVE_SEARCH_MODE == 'grid':
    grid_size = 1
    for values in param_space.values():
        grid_size *= len(values)
    print('Grid combinations:', grid_size)
else:
    print('Bayesian iterations:', n_iter)

print('Active search mode:', _ACTIVE_SEARCH_MODE)
print('Parameter keys:', len(param_space))

## 6. Search & Training Execution

Run the hyperparameter search. This will train multiple TCN configurations.

In [8]:
if search_mode == 'bayesian':
    if not SKOPT_AVAILABLE:
        raise ImportError(
            "Bayesian search requires scikit-optimize. "
            "Install with: pip install scikit-optimize"
        )

    search = BayesSearchCV(
        estimator=model,
        search_spaces=param_space,
        n_iter=n_iter,
        scoring=scorer,
        cv=cv_object,
        n_jobs=1,
        random_state=random_state,
        verbose=verbose,
        return_train_score=True,
        error_score=0.0,
    )
else:
    search = GridSearchCV(
        estimator=model,
        param_grid=param_space,
        scoring=scorer,
        cv=cv_object,
        n_jobs=1,
        verbose=verbose,
        return_train_score=True,
        error_score=0.0,
    )

print(f'Starting {search_mode.upper()} search...')
search.fit(X_train, y_train, groups=groups)

print('\nBest CV score:', search.best_score_)
print('Best params:', search.best_params_)

Starting GRID search...
Fitting 1 folds for each of 32 candidates, totalling 32 fits

Epoch 1/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 14s 14s/step - accuracy: 0.1875 - loss: 2.9769 - val_accuracy: 0.5000 - val_loss: 1.3785
Epoch 2/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 168ms/step - accuracy: 0.3125 - loss: 2.2378 - val_accuracy: 0.5000 - val_loss: 1.3572
Epoch 3/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 163ms/step - accuracy: 0.1875 - loss: 2.5949 - val_accuracy: 0.5000 - val_loss: 1.2759
Epoch 4/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 167ms/step - accuracy: 0.1250 - loss: 2.1440 - val_accuracy: 0.5000 - val_loss: 1.1640
Epoch 5/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 164ms/step - accuracy: 0.3125 - loss: 2.4869 - val_accuracy: 0.5000 - val_loss: 1.0867
Restoring model weights from the end of the best epoch: 5.
Epoch 1/5


KeyboardInterrupt: 

## 7. Holdout Evaluation & Saving Results

In [ ]:
best_model = search.best_estimator_

y_pred = best_model.predict(X_test)

eval_results = evaluate_holdout(
    y_test,
    y_pred,
    target_col=TARGET_COL,
    verbose=True,
)

print('Holdout competition score:', eval_results['competition_score'])

cv_df = pd.DataFrame(search.cv_results_)
cv_df.to_csv(results_dir / f'tcn_level1_cv_{timestamp}.csv', index=False)

eval_results['results_df'].to_csv(
    results_dir / f'tcn_level1_holdout_{timestamp}.csv',
    index=False,
)

pd.DataFrame(
    [
        {
            'best_score': search.best_score_,
            'best_params': str(search.best_params_),
            'holdout_score': eval_results['competition_score'],
        }
    ]
).to_csv(
    results_dir / f'tcn_level1_best_{timestamp}.csv',
    index=False,
)

model_path = results_dir / f'tcn_level1_model_{timestamp}.keras'
try:
    if hasattr(best_model, 'model_') and best_model.model_ is not None:
        best_model.model_.save(model_path)
        print(f'TF model saved to {model_path}')
except Exception as e:
    print(f'Could not save TF model directly: {e}')

## 8. Action Items for Full Training

1. **Increase Epochs:** Change `epochs=5` to `epochs=50` (or higher) in the Model Definition cell.
2. **Full Data:** Set `use_sample_data = False` to use `train.csv`.
3. **Search Mode:** Switch `search_mode = 'bayesian'` and increase `n_iter = 30` (or use `search_mode = 'grid'` with `GRID_PARAM_SPACE`).
4. **Sequence Window:** Tune `extractor__maxlen` / `extractor__chunk_window_size` so they cover the 95th percentile of sequence lengths in `train.csv`.
5. **Feature Extraction:** Bayesian search explores the full extractor space; grid search uses the compact `GRID_PARAM_SPACE` subset.